In [ ]:
%load_ext rich

%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
import unicodedata
from collections import Counter
from operator import itemgetter
from pathlib import Path
from typing import Iterable

import requests
from tqdm import tqdm

import numpy as np
import pandas as pd
import requests
from more_itertools import flatten, unique_everseen
from rapidfuzz import fuzz, process

from aymurai.llm_providers import OllamaLLMProvider
from aymurai.meta.entities import CanonicalEntities, CanonicalEntity
from aymurai.utils.json_data import get_pretty, save_json, load_json

## /document-extract endpoint output

In [ ]:
API_URL = os.getenv("DOCUMENT_API_BASE_URL", "http://127.0.0.1:8000")
ENDPOINT = f"{API_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/disambiguation-eval/files")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT_S = float(os.getenv("DOCUMENT_REQUEST_TIMEOUT", "30"))

print(f"Target endpoint: {ENDPOINT}")
print(f"Data root: {DATA_ROOT.resolve()}")

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {
        "file": (file_path.name, file_path.open("rb"), mime_type),
    }

    try:
        start = time.perf_counter()
        response = session.post(
            ENDPOINT,
            files=files,
            timeout=REQUEST_TIMEOUT_S,
        )
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

## Inference

In [ ]:
# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample})
    response.raise_for_status()
    return response.json()

In [ ]:
from itertools import chain
from operator import itemgetter
from typing import Any

from more_itertools import unique_everseen


def parse_prediction_labels(predictions: list[dict[str, Any]]) -> list[dict[str, str]]:
    """
    Parse prediction labels to extract unique aymurai_label and aymurai_alt_text pairs.

    Args:
        predictions (list[dict[str, Any]]): A list of prediction dictionaries.

    Returns:
        list[dict[str, str]]: A list of dictionaries containing unique aymurai_label and aymurai_alt_text pairs.
    """
    attrs_stream = (
        label.get("attrs") or {}
        for label in chain.from_iterable(pred.get("labels", ()) for pred in predictions)
    )

    unique_pairs = unique_everseen(
        (
            attrs.get("aymurai_label"),
            attrs.get("aymurai_alt_text"),
        )
        for attrs in attrs_stream
        if attrs.get("aymurai_label") and attrs.get("aymurai_alt_text")
    )

    return sorted(
        ({"aymurai_label": label, "text": text} for label, text in unique_pairs),
        key=itemgetter("aymurai_label", "text"),
    )

# 8241_90_17_10_2025_128 CAUSA 14663_2020-2

In this case, I took the .json file that Juli passed me and made a first manual correction of the clustering he did in the [notebook 04 pipeline](04-entity-disambiguation-from-pre-clustered-validations.ipynb).

For this .json, a function needs to be created to ensure that, instead of grouping the files into a single canonical entity, it divides each one into a different canonical entity of the type ***NOMBRE_ARCHIVO***.

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(documents[1]))[0]
)

In [ ]:
# Prepare json to review
json_path = "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
target_path = json_path + target_filename + "-canonical-entities.json"
canonical_entities_to_review = load_json(target_path)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities_to_review = [
    CanonicalEntity.model_validate(entity) for entity in canonical_entities_to_review
]
canonical_entities_to_review = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities_to_review
]

canonical_entities_to_review

In [ ]:
from copy import deepcopy

def split_aliases_by_label(items: list[CanonicalEntity], target_label: str) -> list[CanonicalEntity]:
    """
    Splits aliases into new CanonicalEntities if the item matches target_label.
    Removes non-matching aliases from the original entity.
    """
    processed_items = []

    for item in items:
        # pass through items that don't match the target label
        if item['aymurai_label'] != target_label:
            processed_items.append(item)
            continue

        # if it matches, we process the aliases
        kept_aliases = []
        new_entities = []

        for alias in item['aliases']:
            if alias == item['canonical_text']:
                # Keep this alias in the original item
                kept_aliases.append(alias)
            else:
                # Create a NEW entity for this alias
                new_ce = CanonicalEntity(
                    aymurai_label=target_label,
                    canonical_text=alias,               # The alias becomes the new canonical text
                    aliases=[alias],                    # New entity starts with no aliases
                    attributes=deepcopy(item['attributes']),
                    relations=deepcopy(item['relations'])
                )
                new_entities.append(new_ce)

        # update the original item's aliases
        item['aliases'] = kept_aliases
        
        # add the original item to the result
        processed_items.append(item)
        
        # add the newly created entities to the result
        processed_items.extend(new_entities)

    return processed_items

In [ ]:
canonical_entities_to_review = split_aliases_by_label(canonical_entities_to_review, target_label='NOMBRE_ARCHIVO')

In [ ]:
# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in canonical_entities_to_review
]
canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

## CanonicalEntity extraction

In [ ]:
# Available models
model = "phi4:14b"

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=model)
provider.generate("hola, ¿cómo estás?")

In [ ]:
try:
    stream_prompt = "hola, ¿cómo estás?"
    pieces = []
    for event in provider.stream(stream_prompt):
        print(event.text, end="", flush=True)
        pieces.append(event.text)
    print()
    full_text = "".join(pieces)

except Exception as exc:
    print("La transmisión no está disponible:", exc)

In [ ]:
system_prompt = """
Eres un asistente especializado en anonimización de sentencias judiciales.
Tu tarea es agrupar las menciones de entidades nombradas detectadas por un modelo de NER en **entidades canónicas** sin omitir ninguna mención válida.
Una **entidad canónica** es la representación única de una entidad real.
Agrupa todas las menciones textuales (aliases) que se refieren a una misma persona, documento, lugar, número, fecha, etc., aunque aparezcan con variantes ortográficas o abreviadas.

# Reglas
- Usa solo información presente en el documento y en las menciones del NER; no inventes datos ni concluyas hechos no expresos.
- Toda mención listada por el NER debe evaluarse. Si representa una entidad real, inclúyela en alguna entidad canónica.
- Puede haber falsos positivos y/o negativos, menciones ambiguas o incompletas, por lo que debes evaluar cada mención cuidadosamente.
- Cada entidad canónica debe incluir:
  - `aymurai_label` (etiqueta del NER, p. ej. "PER", "DNI", etc.).
  - `canonical_text` (forma normalizada: nombres completos, fechas normalizadas, etc.).
  - `aliases` (todas las menciones textuales relevantes).
  - `attributes` (diccionario opcional con roles u otras notas, p. ej. {"role": "Juez/a"}).

# Notas
- `aliases` debe conservar las menciones textuales tal como aparecen en el documento.
- `attributes` es especialmente útil para distinguir roles procesales. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"

# Ejemplo
Q: Con fecha 3 de marzo de 2023, la Sra. Laura Beatriz Gómez, DNI 42.987.654, con domicilio en calle Falsa 123, Barrio Los Pinos, Villa Azul, denunció a su expareja, el Sr. Martín Alberto Rodríguez, DNI 27.654.321, con domicilio en calle Real 789, por hechos de violencia física, psicológica y amenazas con armas blancas.
A: ```json
[
  {
    "aymurai_label": "PER",
    "canonical_text": "Laura Beatriz Gómez",
    "aliases": ["Laura Beatriz Gómez"],
    "attributes": {"role": "Denunciante"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "42987654",
    "aliases": ["42.987.654"]
  },
  {
    "aymurai_label": "DIRECCION",
    "canonical_text": "calle Falsa 123",
    "aliases": ["calle Falsa 123"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "Barrio Los Pinos, Villa Azul",
    "aliases": ["Barrio Los Pinos, Villa Azul"]
  },
  {
    "aymurai_label": "PER",
    "canonical_text": "Martín Alberto Rodríguez",
    "aliases": ["Martín Alberto Rodríguez"],
    "attributes": {"role": "Denunciado"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "27654321",
    "aliases": ["27.654.321"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "calle Real 789",
    "aliases": ["calle Real 789"]
  }
]
```
"""

In [ ]:
user_prompt_template = """
A continuación se proporciona un documento judicial y las menciones de entidades nombradas detectadas por un modelo de NER.

# Documento
{document_text}

# Menciones de entidades canónicas agrupadas
{canonical_entities}

# Instrucciones
1. Evalúa cada entidad canónica listada, poniendo especial atención en las aliases.
2. Si alguna alias no corresponde a esa entidad, elimínala de la lista de aliases y créala como una nueva entidad canónica.
3. Las diferencias textuales correctas en los aliases pueden ser variantes ortográficas, abreviaciones, iniciales o errores tipográficos.
4. Diferencias sutiles que permiten desambiguar entidades distintas (por ejemplo, nombres similares de personas diferentes, fechas próximas pero distintas, números de documentos, códigos de identificación o nombres de archivos con pequeñas variaciones, etc.).
5. Normaliza canonical_text; mantén los aliases tal como aparecen.
6. Identifica roles u otros atributos relevantes en el campo attributes (por ejemplo, {{"role": "Denunciante"}}).
"""

In [ ]:
# Extract document
session = requests.Session()
document = call_extraction_api(session, Path(documents[1]))
document = document.get("detail", {}).get("document")

if not document:
    raise ValueError("Document text is empty or not found.")

In [ ]:
# Get paragraphs with at least one NER predicted label
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
pred_paragraphs = [paragraph['document'] for paragraph in ner_predictions if paragraph['labels']]

In [ ]:
# Remove empty 'entity_id', 'relations' and 'attributes' fields
canonical_entities = [
    {
        k: v
        for k, v in ce.items()
        if k not in ("entity_id", "relations", "attributes") or v
    }
    for ce in canonical_entities
]

# Prepare user prompt
user_prompt = user_prompt_template.format(
    document_text="\n".join(pred_paragraphs).strip(),
    canonical_entities=get_pretty(canonical_entities),
)

print(user_prompt)

In [ ]:
try:
    stream_prompt = messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    pieces = []
    for event in provider.stream(
        messages=stream_prompt,
        options={"temperature": 0, "num_ctx": 9_500},
        format=CanonicalEntities.model_json_schema(),
        ):
        print(event.text, end="", flush=True)
        pieces.append(event.text)
    print()
    full_text = "".join(pieces)

except Exception as exc:
    print("La transmisión no está disponible:", exc)

In [ ]:
# Get canonical entities from the model

provider = OllamaLLMProvider(model=model)
response = provider.generate(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    options={"temperature": 0, "num_ctx": 16_384},
    format=CanonicalEntities.model_json_schema(),
)

In [ ]:
canonical_entities = [
        output for output in json.loads(response.text)["canonical_entities"]
    ]

canonical_entities = [
    CanonicalEntity.model_validate(canonical_entity)
    for canonical_entity in canonical_entities
]

In [ ]:
# Save LLM-predicted canonical entities
os.makedirs("../../../resources/data/restricted/disambiguation-eval/canonical-entities/predicted/", exist_ok=True)    

target_path = Path(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/predicted/"
    + target_filename
    + "-canonical-entities.json"
)

save_json(
    [
        canonical_entity.model_dump()
        | {"entity_id": canonical_entity.entity_id.hex}
        for canonical_entity in canonical_entities
    ],
    str(target_path),
)

# 01 - NN x 5C

# Pipeline NER - Clusterization - Validation

In [ ]:
# Extract document
session = requests.Session()
document = call_extraction_api(session, Path(documents[0]))


In [ ]:
document = document.get("detail", {}).get("document")

if not document:
    raise ValueError("Document text is empty or not found.")

# Get NER predictions
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
parsed_ner_labels = parse_prediction_labels(ner_predictions)
ner_output_json = get_pretty(parsed_ner_labels)

In [ ]:
def normalize(s: str) -> str:
    """
    Normalize string for clustering.

    Args:
        s (str): input string

    Returns:
        str: normalized string
    """
    # strip accents, lowercase, collapse spaces
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
    s = " ".join(s.lower().split())
    return s


def cluster_with_cdist(
    entities: list[str], threshold: int = 90, scorer: callable = fuzz.token_set_ratio
):
    """
    Cluster entities based on similarity using a distance matrix.

    Args:
        entities (list[str]): list of entity strings
        threshold (int): similarity threshold for clustering
        scorer (callable): similarity scoring function

    Returns:
        list[list[tuple[str, str]]]: clusters of (original, normalized) entity tuples
    """
    # matrix of similarities on normalized strings
    # normalize however you like; here just lower + strip
    normed = [" ".join(e.lower().split()) for e in entities]
    sim = process.cdist(normed, normed, scorer=scorer, score_cutoff=threshold)
    sim = np.array(sim)

    # union-find
    parent = list(range(len(normed)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)
        if ri != rj:
            parent[rj] = ri

    # link pairs above threshold (upper triangle only)
    n = len(normed)
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= threshold:
                union(i, j)

    # collect clusters
    clusters = {}
    for idx in range(n):
        root = find(idx)
        clusters.setdefault(root, []).append((entities[idx], normed[idx]))
    return list(clusters.values())


def pick_canonical(cluster: list[tuple[str, str]]) -> str:
    """
    Pick canonical text from a cluster.

    Args:
        cluster (list[tuple[str, str]]): cluster of (original, normalized) entity tuples

    Returns:
        str: chosen canonical text
    """
    # prefer longest original; fall back to first
    return max(cluster, key=lambda x: len(x[0]))[0]

In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in parsed_ner_labels],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
def parse_item(item: tuple[str, ...]) -> tuple[str, str, str]:
    """
    Parse an item into (label, orig, norm).

    Accepts:
      - (orig, norm, label)
      - (labelled_orig, labelled_norm) with prefix 'LABEL:'
    Args:
        item (tuple[str, ...]): input item

    Returns:
        tuple[str, str, str]: (label, orig, norm)
    """
    if len(item) == 3:
        orig, norm, label = item
        return label, orig, norm

    # len == 2: assume "LABEL:text"
    labelled_orig, labelled_norm = item
    label, orig = labelled_orig.split(":", 1)
    _, norm = labelled_norm.split(":", 1)
    return label, orig, norm


def pick_cluster_label(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Pick the most common label from parsed items.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen label
    """
    labels = [lbl for lbl, _, _ in parsed_items]
    # majority vote; fallback to first
    return Counter(labels).most_common(1)[0][0]


def pick_canonical_text(parsed_items: list[tuple[str, str, str]]) -> str:
    """
    Choose the longest original surface form; tweak as needed.

    Args:
        parsed_items (list[tuple[str, str, str]]): parsed items

    Returns:
        str: chosen canonical text
    """
    return max(parsed_items, key=lambda x: len(x[1]))[1]


def clusters_to_canonical_entities(
    clusters: list[list[tuple[str, str]]],
) -> list[CanonicalEntity]:
    """
    Convert clusters to CanonicalEntity objects.

    Args:
        clusters (list[list[tuple[str, str]]]): clusters of (original, normalized) entity tuples

    Returns:
        list[CanonicalEntity]: list of CanonicalEntity objects
    """
    canonical_entities = []

    for cluster in clusters:
        parsed = [parse_item(item) for item in cluster]  # [(label, orig, norm), ...]
        label = pick_cluster_label(parsed)
        canonical_text = pick_canonical_text(parsed)
        aliases = sorted({orig for _, orig, _ in parsed})
        ce = CanonicalEntity(
            aymurai_label=label,
            canonical_text=canonical_text,
            aliases=aliases,
            attributes={},
            relations=[],
        )
        canonical_entities.append(ce)

    return canonical_entities

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)

# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(documents[0]))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

In [ ]:
target_filename

In [ ]:

# Save canonical entities to JSON in to-review folder to save the raw output
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/to-review/"
    + target_filename
    + "-canonical-entities.json",
)

# Save canonical entities to JSON in reviewing folder to make te review manually
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json",
)

Make a pause and review the json manually for those wrong outputs

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

## CanonicalEntity extraction

In [ ]:
# Available models
model = "phi4:14b"

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=model)
provider.generate("hola, ¿cómo estás?")

In [ ]:
system_prompt = """
Eres un asistente especializado en anonimización de sentencias judiciales.
Tu tarea es agrupar las menciones de entidades nombradas detectadas por un modelo de NER en **entidades canónicas** sin omitir ninguna mención válida.
Una **entidad canónica** es la representación única de una entidad real.
Agrupa todas las menciones textuales (aliases) que se refieren a una misma persona, documento, lugar, número, fecha, etc., aunque aparezcan con variantes ortográficas o abreviadas.

# Reglas
- Usa solo información presente en el documento y en las menciones del NER; no inventes datos ni concluyas hechos no expresos.
- Toda mención listada por el NER debe evaluarse. Si representa una entidad real, inclúyela en alguna entidad canónica.
- Puede haber falsos positivos y/o negativos, menciones ambiguas o incompletas, por lo que debes evaluar cada mención cuidadosamente.
- Cada entidad canónica debe incluir:
  - `aymurai_label` (etiqueta del NER, p. ej. "PER", "DNI", etc.).
  - `canonical_text` (forma normalizada: nombres completos, fechas normalizadas, etc.).
  - `aliases` (todas las menciones textuales relevantes).
  - `attributes` (diccionario opcional con roles u otras notas, p. ej. {"role": "Juez/a"}).

# Notas
- `aliases` debe conservar las menciones textuales tal como aparecen en el documento.
- `attributes` es especialmente útil para distinguir roles procesales. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"

# Ejemplo
Q: Con fecha 3 de marzo de 2023, la Sra. Laura Beatriz Gómez, DNI 42.987.654, con domicilio en calle Falsa 123, Barrio Los Pinos, Villa Azul, denunció a su expareja, el Sr. Martín Alberto Rodríguez, DNI 27.654.321, con domicilio en calle Real 789, por hechos de violencia física, psicológica y amenazas con armas blancas.
A: ```json
[
  {
    "aymurai_label": "PER",
    "canonical_text": "Laura Beatriz Gómez",
    "aliases": ["Laura Beatriz Gómez"],
    "attributes": {"role": "Denunciante"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "42987654",
    "aliases": ["42.987.654"]
  },
  {
    "aymurai_label": "DIRECCION",
    "canonical_text": "calle Falsa 123",
    "aliases": ["calle Falsa 123"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "Barrio Los Pinos, Villa Azul",
    "aliases": ["Barrio Los Pinos, Villa Azul"]
  },
  {
    "aymurai_label": "PER",
    "canonical_text": "Martín Alberto Rodríguez",
    "aliases": ["Martín Alberto Rodríguez"],
    "attributes": {"role": "Denunciado"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "27654321",
    "aliases": ["27.654.321"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "calle Real 789",
    "aliases": ["calle Real 789"]
  }
]
```
"""

In [ ]:
user_prompt_template = """
A continuación se proporciona un documento judicial y las menciones de entidades nombradas detectadas por un modelo de NER.

# Documento
{document_text}

# Menciones de entidades canónicas agrupadas
{canonical_entities}

# Instrucciones
1. Evalúa cada entidad canónica listada, poniendo especial atención en las aliases.
2. Si alguna alias no corresponde a esa entidad, elimínala de la lista de aliases y créala como una nueva entidad canónica.
3. Las diferencias textuales correctas en los aliases pueden ser variantes ortográficas, abreviaciones, iniciales o errores tipográficos.
4. Diferencias sutiles que permiten desambiguar entidades distintas (por ejemplo, nombres similares de personas diferentes, fechas próximas pero distintas, números de documentos, códigos de identificación o nombres de archivos con pequeñas variaciones, etc.).
5. Normaliza canonical_text; mantén los aliases tal como aparecen.
6. Identifica roles u otros atributos relevantes en el campo attributes (por ejemplo, {{"role": "Denunciante"}}).
"""

In [ ]:
# Extract document
session = requests.Session()
document = call_extraction_api(session, Path(documents[0]))
document = document.get("detail", {}).get("document")

if not document:
    raise ValueError("Document text is empty or not found.")

In [ ]:
# Get paragraphs with at least one NER predicted label
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
pred_paragraphs = [paragraph['document'] for paragraph in ner_predictions if paragraph['labels']]

In [ ]:
# Remove empty 'entity_id', 'relations' and 'attributes' fields
canonical_entities = [
    {
        k: v
        for k, v in ce.items()
        if k not in ("entity_id", "relations", "attributes") or v
    }
    for ce in canonical_entities
]

# Prepare user prompt
user_prompt = user_prompt_template.format(
    document_text="\n".join(document).strip(),
    canonical_entities=get_pretty(canonical_entities),
)

print(user_prompt)

In [ ]:
# Get canonical entities from the model
provider = OllamaLLMProvider(model=model)
response = provider.generate(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    options={"temperature": 0, "num_ctx": 9_500},
    format=CanonicalEntities.model_json_schema(),
)

In [ ]:
canonical_entities = [
        output for output in json.loads(response.text)["canonical_entities"]
    ]

canonical_entities = [
    CanonicalEntity.model_validate(canonical_entity)
    for canonical_entity in canonical_entities
]

In [ ]:
canonical_entities

In [ ]:
# Save LLM-predicted canonical entities
os.makedirs("../../../resources/data/restricted/disambiguation-eval/canonical-entities/predicted/", exist_ok=True)    

save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/predicted/"
    + target_filename
    + "-canonical-entities.json",
)

# 1- FLORES (pdf)

# Pipeline NER - Clusterization - Validation

In [ ]:
json_path = "../../../resources/data/restricted/disambiguation-eval/files/1- FLORES ABARCA, Francisco Alexander s 149 bis J-01-00369422-0-2022-1 Sala Feria (II) nnya vict do apela pp_response_1765835782205.json"

json_path = Path(json_path)

In [ ]:
# Extract document
document = load_json(json_path)["document"]

In [ ]:
if not document:
    raise ValueError("Document text is empty or not found.")

# Get NER predictions
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
parsed_ner_labels = parse_prediction_labels(ner_predictions)
ner_output_json = get_pretty(parsed_ner_labels)

In [ ]:
parsed_ner_labels

In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in parsed_ner_labels],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)

# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(json_path))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

In [ ]:
# Save canonical entities to JSON in to-review folder to save the raw output
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/to-review/"
    + target_filename
    + "-canonical-entities.json",
)

# Save canonical entities to JSON in reviewing folder to make te review manually
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json",
)

Make a pause and review the json manually for those wrong outputs

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

# 2 - FISSCHER (pdf)

# Pipeline NER - Clusterization - Validation

In [ ]:
json_path = "../../../resources/data/restricted/disambiguation-eval/files/2 - FISSCHER Alejandro Claudio s - 92_response_1765908855010.json"

json_path = Path(json_path)

In [ ]:
# Extract document
document = load_json(json_path)["document"]

In [ ]:
if not document:
    raise ValueError("Document text is empty or not found.")

# Get NER predictions
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
parsed_ner_labels = parse_prediction_labels(ner_predictions)
ner_output_json = get_pretty(parsed_ner_labels)

In [ ]:
parsed_ner_labels

In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in parsed_ner_labels],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)

# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(json_path))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

In [ ]:
# Save canonical entities to JSON in to-review folder to save the raw output
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/to-review/"
    + target_filename
    + "-canonical-entities.json",
)

# Save canonical entities to JSON in reviewing folder to make te review manually
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json",
)

Make a pause and review the json manually for those wrong outputs

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

# 2 - Feb. 2 (pdf)

# Pipeline NER - Clusterization - Validation

In [ ]:
json_path = "../../../resources/data/restricted/disambiguation-eval/files/2 - Feb. 2 - DE ON, Kwan x 13944_response_1765836659235.json"

json_path = Path(json_path)

In [ ]:
# Extract document
document = load_json(json_path)["document"]

In [ ]:
if not document:
    raise ValueError("Document text is empty or not found.")

# Get NER predictions
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
parsed_ner_labels = parse_prediction_labels(ner_predictions)
ner_output_json = get_pretty(parsed_ner_labels)

In [ ]:
parsed_ner_labels

In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in parsed_ner_labels],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)

# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(json_path))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

In [ ]:
# Save canonical entities to JSON in to-review folder to save the raw output
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/to-review/"
    + target_filename
    + "-canonical-entities.json",
)

# Save canonical entities to JSON in reviewing folder to make te review manually
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json",
)

Make a pause and review the json manually for those wrong outputs

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

# Jul. 18 (pdf)

# Pipeline NER - Clusterization - Validation

In [ ]:
json_path = "../../../resources/data/restricted/disambiguation-eval/files/Jul. 18- SALA IV- IPANAQUE VILLAR Carlos Hector s 5c nnapes_response_1765835513742.json"

json_path = Path(json_path)

In [ ]:
# Extract document
document = load_json(json_path)["document"]

In [ ]:
if not document:
    raise ValueError("Document text is empty or not found.")

# Get NER predictions
ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
parsed_ner_labels = parse_prediction_labels(ner_predictions)
ner_output_json = get_pretty(parsed_ner_labels)

In [ ]:
parsed_ner_labels

In [ ]:
# Cluster entities
clusters = cluster_with_cdist(
    [f"{v['aymurai_label']}:{v['text']}" for v in parsed_ner_labels],
    threshold=95,
)

for cluster in clusters:
    canonical = pick_canonical(cluster)
    originals = [c[0] for c in cluster]
    print(f"Canonical: {canonical} -> {originals}")

In [ ]:
canonical_entities = clusters_to_canonical_entities(clusters)

# Sort by label and canonical text
canonical_entities = sorted(
    canonical_entities,
    key=lambda ce: (ce.aymurai_label.lower(), ce.canonical_text.lower()),
)
canonical_entities

In [ ]:
# Prepare target filename
target_filename = re.sub(
    r"\s+|_", "-", os.path.splitext(os.path.basename(json_path))[0]
)
target_filename = re.sub(r"-{2,}", "-", target_filename)

In [ ]:
# Save canonical entities to JSON in to-review folder to save the raw output
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/to-review/"
    + target_filename
    + "-canonical-entities.json",
)

# Save canonical entities to JSON in reviewing folder to make te review manually
save_json(
    [ce.model_dump() | {"entity_id": None} for ce in canonical_entities],
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json",
)

Make a pause and review the json manually for those wrong outputs

In [ ]:
# Execute this cell to load reviewed canonical entities
reviewed_canonical_entities = load_json(
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewing/"
    + target_filename
    + "-canonical-entities.json"
)

# Convert reviewed entities back to CanonicalEntity objects
canonical_entities = [
    CanonicalEntity.model_validate(entity) for entity in reviewed_canonical_entities
]

canonical_entities = [
    entity.model_dump() | {"entity_id": entity.entity_id.hex}
    for entity in canonical_entities
]

canonical_entities

In [ ]:
# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/reviewed/"
    + target_filename
    + "-canonical-entities.json",
)

# Save reviewed canonical entities
save_json(
    canonical_entities,
    "../../../resources/data/restricted/disambiguation-eval/canonical-entities/manual-predicted/"
    + target_filename
    + "-canonical-entities.json",
)

# Start point of the old version of the pipeline

## CanonicalEntity extraction

In [ ]:
# Available models
# model = "gpt-oss:20b"
# model = "llama3"
model = "phi4:14b"
# model = "llama3.1:8b"
# model = "gemma3:270m"

In [ ]:
# Sanity check
provider = OllamaLLMProvider(model=model)
provider.generate("hola, ¿cómo estás?")

In [ ]:
try:
    stream_prompt = "hola, ¿cómo estás?"
    pieces = []
    for event in provider.stream(stream_prompt):
        print(event.text, end="", flush=True)
        pieces.append(event.text)
    print()
    full_text = "".join(pieces)

except Exception as exc:
    print("La transmisión no está disponible:", exc)

In [ ]:
system_prompt = """
Eres un asistente especializado en anonimización de sentencias judiciales.
Tu tarea es agrupar las menciones de entidades nombradas detectadas por un modelo de NER en **entidades canónicas** sin omitir ninguna mención válida.
Una **entidad canónica** es la representación única de una entidad real.
Agrupa todas las menciones textuales (aliases) que se refieren a una misma persona, documento, lugar, número, fecha, etc., aunque aparezcan con variantes ortográficas o abreviadas.

# Reglas
- Usa solo información presente en el documento y en las menciones del NER; no inventes datos ni concluyas hechos no expresos.
- Toda mención listada por el NER debe evaluarse. Si representa una entidad real, inclúyela en alguna entidad canónica.
- Puede haber falsos positivos y/o negativos, menciones ambiguas o incompletas, por lo que debes evaluar cada mención cuidadosamente.
- Cada entidad canónica debe incluir:
  - `aymurai_label` (etiqueta del NER, p. ej. "PER", "DNI", etc.).
  - `canonical_text` (forma normalizada: nombres completos, fechas normalizadas, etc.).
  - `aliases` (todas las menciones textuales relevantes).
  - `attributes` (diccionario opcional con roles u otras notas, p. ej. {"role": "Juez/a"}).

# Notas
- `aliases` debe conservar las menciones textuales tal como aparecen en el documento.
- `attributes` es especialmente útil para distinguir roles procesales. Entre ellos pueden incluirse:
    * "Denunciante"
    * "Denunciado/a"
    * "Víctima"
    * "Juez/a"
    * "Fiscal"
    * "Abogado/a"
    * "Perito/a"
    * "Testigo"

# Ejemplo
Q: Con fecha 3 de marzo de 2023, la Sra. Laura Beatriz Gómez, DNI 42.987.654, con domicilio en calle Falsa 123, Barrio Los Pinos, Villa Azul, denunció a su expareja, el Sr. Martín Alberto Rodríguez, DNI 27.654.321, con domicilio en calle Real 789, por hechos de violencia física, psicológica y amenazas con armas blancas.
A: ```json
[
  {
    "aymurai_label": "PER",
    "canonical_text": "Laura Beatriz Gómez",
    "aliases": ["Laura Beatriz Gómez"],
    "attributes": {"role": "Denunciante"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "42987654",
    "aliases": ["42.987.654"]
  },
  {
    "aymurai_label": "DIRECCION",
    "canonical_text": "calle Falsa 123",
    "aliases": ["calle Falsa 123"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "Barrio Los Pinos, Villa Azul",
    "aliases": ["Barrio Los Pinos, Villa Azul"]
  },
  {
    "aymurai_label": "PER",
    "canonical_text": "Martín Alberto Rodríguez",
    "aliases": ["Martín Alberto Rodríguez"],
    "attributes": {"role": "Denunciado"}
  },
  {
    "aymurai_label": "DNI",
    "canonical_text": "27654321",
    "aliases": ["27.654.321"]
  },
  {
    "aymurai_label": "LOC",
    "canonical_text": "calle Real 789",
    "aliases": ["calle Real 789"]
  }
]
```
"""


In [ ]:
user_prompt_template = """
A continuación se proporciona un documento judicial y las menciones de entidades nombradas detectadas por un modelo de NER.

# Documento
{document_text}

# Menciones de entidades detectadas por el NER
{ner_output_json}

# Instrucciones
1. Evalúa cada mención listada. Si representa una entidad real, inclúyela en la lista de aliases de alguna entidad canónica.
2. No omitas entidades válidas, alias con iniciales ni variantes abreviadas.
3. Solo fusiona menciones cuando exista evidencia clara de que se refieren a la misma entidad.
4. Normaliza canonical_text; mantén los alias tal como aparecen.
5. Utiliza attributes para indicar roles (p. ej. {{"role": "Denunciante"}}) o aclaraciones de desambiguación.
"""

In [ ]:
def extract_canonical_entities(
    doc_path: str, model: str = "phi4:14b"
) -> CanonicalEntities:
    """
    Extract canonical entities from a document.

    Args:
        doc_path (str): The path to the document.
        model (str, optional): The model to use for extraction. Defaults to "phi4:14b".

    Raises:
        ValueError: If the document is empty or not found.
        ValueError: If the document summary is empty or not found.

    Returns:
        CanonicalEntities: The extracted canonical entities.
    """
    # Extract document
    session = requests.Session()
    document = call_extraction_api(session, Path(doc_path))
    document = document.get("detail", {}).get("document")

    if not document:
        raise ValueError("Document text is empty or not found.")

    # Get NER predictions
    ner_predictions = [get_predictions(paragraph) for paragraph in tqdm(document)]
    parsed_ner_labels = parse_prediction_labels(ner_predictions)
    ner_output_json = get_pretty(parsed_ner_labels)

    # Prepare user prompt
    user_prompt = user_prompt_template.format(
        document_text="\n".join(document).strip(),
        ner_output_json=ner_output_json,
    )

    # Get canonical entities from the model
    provider = OllamaLLMProvider(model=model)
    response = provider.generate(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={"temperature": 0, "num_ctx": 12_000},
        format=CanonicalEntities.model_json_schema(),
    )

    # Parse canonical entities from response
    canonical_entities = [
        output for output in json.loads(response.text)["canonical_entities"]
    ]

    canonical_entities = [
        CanonicalEntity.model_validate(canonical_entity)
        for canonical_entity in canonical_entities
    ]

    return canonical_entities

In [ ]:
output_root = Path(DATA_ROOT) / "preds"
output_root.mkdir(parents=True, exist_ok=True)

prompt_version = "1"

timestamp = time.strftime("%y%m%d_%H%M")
base_dir_name = f"preds-{model}-pv{prompt_version}-{timestamp}"
output_dir = output_root / base_dir_name

output_dir.mkdir(parents=True, exist_ok=False)
print(f"Saving canonical entities to {output_dir}")

for doc_path in documents:
    print(f"Processing document: {doc_path}")

    try:
        canonical_entities = extract_canonical_entities(doc_path, model=model)
        print(f"Extracted {len(canonical_entities)} canonical entities.")

        target_filename = re.sub(
            r"\s+|_", "-", os.path.splitext(os.path.basename(doc_path))[0]
        )
        target_filename = re.sub(r"-{2,}", "-", target_filename)
        target_path = output_dir / f"{target_filename}.json"

        save_json(
            [
                canonical_entity.model_dump()
                | {"entity_id": canonical_entity.entity_id.hex}
                for canonical_entity in canonical_entities
            ],
            str(target_path),
        )

    except Exception as e:
        print(f"Error processing document {doc_path}: {e}")